# 03 – Preprocessing & Feature Engineering

**Lokal oder Colab** — gleiche Pipeline, nur Pfade unterschiedlich:

| | **Lokal** | **Colab** |
|---|-----------|-----------|
| Code | Repo-Clone (`scripts/`) | `git pull` → `/content/DataMining_Final-Project` |
| CSVs | `data/train.csv`, `data/test.csv` | dieselben Dateien auf Drive unter `MyDrive/DataMining/DataMining_Final-Project/data/` |
| Outputs | `outputs/processed/` | `MyDrive/.../outputs/processed/` |

[Colab öffnen](https://colab.research.google.com/github/jspldrch/DataMining_Final-Project/blob/main/notebooks/03_preprocessing.ipynb) → Runtime **CPU** → Run all.

`MODE=full` (wenn `train.csv` da): Streaming pro Region (~20–40 Min.). Sonst `train_sample.csv` für einen Schnelltest.

→ Notebook 04: `train_labeled.parquet`, `test_features.parquet`

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Colab cold-start: clone repo before importing scripts (Open-in-Colab = nur .ipynb)
_repo = Path("/content/DataMining_Final-Project")
if not (_repo / "scripts" / "notebook_init.py").exists():
    try:
        import google.colab  # noqa: F401
        import subprocess
        subprocess.run(
            ["git", "clone", "--branch", "main",
             "https://github.com/jspldrch/DataMining_Final-Project.git", str(_repo)],
            check=True,
        )
    except ImportError:
        pass
for _p in (Path.cwd(), Path.cwd().parent, _repo):
    if (_p / "scripts" / "notebook_init.py").exists() and str(_p.resolve()) not in sys.path:
        sys.path.insert(0, str(_p.resolve()))

from scripts.notebook_init import setup
from scripts.project_env import load_script

env = setup()
PROJECT_ROOT = env["project_root"]
DATA_DIR = env["data_dir"]
TRAIN_PATH = env["train_path"]
TEST_PATH = env["test_path"]
PROCESSED_DIR = env["processed_dir"]
MODE = env["mode"]

_features = load_script("dm_features", PROJECT_ROOT / "scripts" / "features.py")
build_features = _features.build_features
feature_columns = _features.feature_columns
add_persistence_baseline = _features.add_persistence_baseline

print("Setup OK | MODE:", MODE)

In [ ]:
path_train_std = PROCESSED_DIR / "train_labeled.parquet"
path_test_std = PROCESSED_DIR / "test_features.parquet"
feat_cols = feature_columns()

if MODE == "full":
    _ps = load_script("preprocess_streaming", PROJECT_ROOT / "scripts" / "preprocess_streaming.py")

    print("Streaming (pro Region) — dauert ~20–40 Min. …")
    stats = _ps.preprocess_by_region(
        TRAIN_PATH, TEST_PATH, path_train_std, path_test_std, chunk_size=500_000
    )
    print("Fertig:", stats)
else:
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    train["_split"] = "train"
    test["_split"] = "test"
    test["score"] = np.nan
    panel = build_features(pd.concat([train, test], ignore_index=True))
    panel = add_persistence_baseline(panel, lag_days=7)

    train_labeled = panel[(panel["_split"] == "train") & panel["score"].notna()]
    test_feat = panel[panel["_split"] == "test"]
    meta = ["region_id", "date", "year", "month", "day", "ordinal", "score", "score_persist7"]
    save_train = [c for c in list(dict.fromkeys(meta + feat_cols)) if c in train_labeled.columns]
    save_test = [c for c in list(dict.fromkeys(
        ["region_id", "date", "year", "month", "day", "ordinal"] + feat_cols
    )) if c in test_feat.columns]
    train_labeled[save_train].to_parquet(path_train_std, index=False)
    test_feat[save_test].to_parquet(path_test_std, index=False)
    print(f"Gespeichert: {len(train_labeled):,} train | {len(test_feat):,} test")

In [ ]:
import pyarrow.parquet as pq

n_train = pq.read_metadata(path_train_std).num_rows
n_test = pq.read_metadata(path_test_std).num_rows
print(f"Parquet: train_labeled {n_train:,} | test {n_test:,}")
pd.read_parquet(path_train_std).head(3)